### Import libraries

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

### Define the file, model and scenarios

In [ ]:
input_file = "data/model_results.xlsx"

selected_model = "ModelName"
baseline_scenario = "BaselineScenario"
policy_scenario = "PolicyScenario"
delayed_policy_scenario = "DelayedPolicyScenario" # Needed for the IT indicator only, and it might not be available

### Define the variables to retain

In [ ]:
variables_to_keep = [
    "Emissions|CO2|Energy and Industrial Processes",
    "Emissions|Kyoto Gases", # if available, can be used in CAV
    "Final Energy",
    "GDP|PPP",
    "GDP|MER",
    "Primary Energy|Fossil",
    "Primary Energy|Non-Biomass Renewables",
    "Primary Energy|Biomass",
    "Primary Energy|Nuclear",
    "Price|Carbon",
    "Policy Cost|GDP Loss", # if available, can be used in CAV
    "Consumption" # if available, can be used in CAV
]

### Load the IAMC dataset and filter variables

In [ ]:
iamc_df = pd.read_excel(input_file)

iamc_df = iamc_df[
    iamc_df["Variable"].isin(variables_to_keep)
].copy()

### Select the model and scenario pair

In [ ]:
diagnostics_df = iamc_df[
    (iamc_df["Model"] == selected_model)
    & iamc_df["Scenario"].isin([
        baseline_scenario,
        policy_scenario,
        delayed_policy_scenario
    ])
].copy()

diagnostics_df.head()

### Prepare the data layout

In [ ]:
if not {"Year", "Value"}.issubset(diagnostics_df.columns): ## Else keep long layout and skip conversion.
    year_columns = [
        c for c in diagnostics_df.columns
        if str(c).isdigit()
    ]

    diagnostics_df = diagnostics_df.melt(
        id_vars=["Model", "Scenario", "Region", "Variable", "Unit"],
        value_vars=year_columns,
        var_name="Year",
        value_name="Value"
    )

diagnostics_df["Year"] = diagnostics_df["Year"].astype(int)
diagnostics_df["Value"] = pd.to_numeric(diagnostics_df["Value"])

diagnostics_df.head()

### Calculate RAI

In [ ]:
RAI_vars = ["Emissions|CO2|Energy and Industrial Processes"]

RAI_df = diagnostics_df[
    diagnostics_df["Variable"].isin(RAI_vars)
].copy()

RAI_df = RAI_df.pivot(
    index=["Model", "Region", "Variable", "Unit", "Year"],
    columns="Scenario",
    values="Value"
).reset_index()

RAI_df["RAI"] = (
    RAI_df[baseline_scenario] - RAI_df[policy_scenario]
) / RAI_df[baseline_scenario]

RAI_df.head()

### Calculate ERT indicator

In [ ]:
ERT_vars = [
    "Emissions|CO2|Energy and Industrial Processes",
    "Final Energy",
    "GDP|PPP"
]

ERT_df = diagnostics_df[
    diagnostics_df["Variable"].isin(ERT_vars)
].copy()

ERT_df = ERT_df.pivot(
    index=["Model", "Region", "Year"],
    columns=["Scenario", "Variable"],
    values="Value"
).reset_index()

for scenario in [baseline_scenario, policy_scenario]:
    ERT_df[(scenario, "CI")] = (
        ERT_df[(scenario, "Emissions|CO2|Energy and Industrial Processes")]
        / ERT_df[(scenario, "Final Energy")]
    )

    ERT_df[(scenario, "EI")] = (
        ERT_df[(scenario, "Final Energy")]
        / ERT_df[(scenario, "GDP|PPP")]
    )

ERT_df["CI_red"] = (
    ERT_df[(baseline_scenario, "CI")] - ERT_df[(policy_scenario, "CI")]
) / ERT_df[(baseline_scenario, "CI")]

ERT_df["EI_red"] = (
    ERT_df[(baseline_scenario, "EI")] - ERT_df[(policy_scenario, "EI")]
) / ERT_df[(baseline_scenario, "EI")]

ERT_df["ERT"] = ERT_df["CI_red"] / (
    ERT_df["CI_red"] + ERT_df["EI_red"]
).replace(0, np.nan)

ERT_df.head()

### Calculate TI indicator

In [ ]:
TI_vars = [
    "Primary Energy|Fossil",
    "Primary Energy|Non-Biomass Renewables",
    "Primary Energy|Biomass",
    "Primary Energy|Nuclear"
]

baseyear = 2020

TI_df = diagnostics_df[
    diagnostics_df["Variable"].isin(TI_vars)
].copy()

TI_df = TI_df.pivot(
    index=["Model", "Scenario", "Region", "Year"],
    columns="Variable",
    values="Value"
).reset_index()

# Convert primary energy values to shares; inputs must use the same unit.
PE_total = TI_df[TI_vars].sum(axis=1, min_count=4)

TI_df[TI_vars] = TI_df[TI_vars].div(
    PE_total.replace(0, np.nan), axis=0
)

# Match each time series with its base-year shares.
group_keys = ["Model", "Scenario", "Region"]

base_shares = (
    TI_df[TI_df["Year"] == baseyear]
    .set_index(group_keys)[TI_vars]
    .add_suffix("_base")
)

TI_df = TI_df.merge(base_shares, on=group_keys, how="left")

# Sum the absolute changes in shares relative to the base year.
TI_df["TI"] = 0.0

for variable in TI_vars:
    TI_df["TI"] += (
        TI_df[variable] - TI_df[f"{variable}_base"]
    ).abs()

TI_df.head()

### Calculate FFR indicator

In [ ]:
FFR_variable = "Primary Energy|Fossil"

baseyear = 2020

FFR_df = diagnostics_df[
    diagnostics_df["Variable"] == FFR_variable
].copy()

FFR_df = FFR_df.pivot(
    index=["Model", "Scenario", "Region", "Year"],
    columns="Variable",
    values="Value"
).reset_index()

group_keys = ["Model", "Scenario", "Region"]

base_fossil = (
    FFR_df[FFR_df["Year"] == baseyear]
    .set_index(group_keys)[[FFR_variable]]
    .add_suffix("_base")
)

FFR_df = FFR_df.merge(base_fossil, on=group_keys, how="left")

FFR_df["FFR"] = (
    FFR_df[f"{FFR_variable}_base"] - FFR_df[FFR_variable]
) / FFR_df[f"{FFR_variable}_base"].replace(0, np.nan)

FFR_df.head()

### Calculate IT Indicator (only if a delayed policy scenario is provided)

In [ ]:
IT_vars = ["Emissions|CO2|Energy and Industrial Processes"]

shockyear = 2030
endyear = 2050

IT_df = diagnostics_df[
    diagnostics_df["Variable"].isin(IT_vars)
    & diagnostics_df["Scenario"].isin([
        policy_scenario,
        delayed_policy_scenario
    ])
].copy()

IT_df = IT_df.pivot(
    index=["Model", "Region", "Variable", "Unit", "Year"],
    columns="Scenario",
    values="Value"
).reset_index()

IT_df["diff"] = (
    IT_df[delayed_policy_scenario] - IT_df[policy_scenario]
)

# Arrange emissions gaps by year.
gaps = IT_df.pivot(
    index=["Model", "Region", "Variable", "Unit"],
    columns="Year",
    values="diff"
)

IT_years = sorted(
    {shockyear, endyear}
    | {year for year in gaps.columns if shockyear <= year <= endyear}
)

gaps = gaps.reindex(columns=IT_years)

# Integrate between reporting years, assuming linear changes.
cumulative_diff = 0

for year_start, year_end in zip(IT_years[:-1], IT_years[1:]):
    cumulative_diff += (
        (gaps[year_start] + gaps[year_end]) / 2
        * (year_end - year_start)
    )

# Normalise by the shock-year emissions gap.
IT_df = gaps[[shockyear]].rename(
    columns={shockyear: "Gap_shockyear"}
)

IT_df["Cumulative_diff"] = cumulative_diff

# IT is expressed in years.
IT_df["IT"] = (
    IT_df["Cumulative_diff"]
    / IT_df["Gap_shockyear"].replace(0, np.nan)
)

IT_df = IT_df.reset_index()

IT_df.head()

### Calculate CAV Indicator

In [ ]:
gdp_variable = "GDP|PPP"  # Or "GDP|MER"

# Adjust to the input units and reporting convention.
money_to_currency = 1e6      # Monetary inputs in million currency/yr
emissions_to_tonnes = 1e6    # Emissions in Mt/yr
reported_loss_sign = -1      # -1 if losses are negative; +1 if positive
reported_loss_is_percent = False  # True if reported as % of baseline GDP

# Monetary inputs and carbon price must use compatible currency valuations.

baseline_vars = diagnostics_df.loc[
    diagnostics_df["Scenario"] == baseline_scenario, "Variable"
].unique()

policy_vars = diagnostics_df.loc[
    diagnostics_df["Scenario"] == policy_scenario, "Variable"
].unique()

# Select emissions coverage.
if (
    "Emissions|Kyoto Gases" in baseline_vars
    and "Emissions|Kyoto Gases" in policy_vars
):
    CAV_emissions = "Emissions|Kyoto Gases"
else:
    CAV_emissions = "Emissions|CO2|Energy and Industrial Processes"

# Select the available cost input.
if "Consumption" in baseline_vars and "Consumption" in policy_vars:
    CAV_cost = "Consumption"
elif "Policy Cost|GDP Loss" in policy_vars:
    CAV_cost = "Policy Cost|GDP Loss"
else:
    CAV_cost = gdp_variable

CAV_vars = [
    CAV_emissions,
    "Price|Carbon",
    CAV_cost,
    gdp_variable
]

CAV_df = diagnostics_df[
    diagnostics_df["Variable"].isin(CAV_vars)
    & diagnostics_df["Scenario"].isin([
        baseline_scenario,
        policy_scenario
    ])
].copy()

CAV_df = CAV_df.pivot(
    index=["Model", "Region", "Year"],
    columns=["Scenario", "Variable"],
    values="Value"
).reset_index()

# Calculate policy cost using the selected input.
if CAV_cost == "Consumption":
    CAV_df["Policy_cost"] = (
        CAV_df[(baseline_scenario, "Consumption")]
        - CAV_df[(policy_scenario, "Consumption")]
    ) * money_to_currency

elif CAV_cost == "Policy Cost|GDP Loss":
    reported_loss = (
        CAV_df[(policy_scenario, "Policy Cost|GDP Loss")]
        * reported_loss_sign
    )

    if reported_loss_is_percent:
        CAV_df["Policy_cost"] = (
            reported_loss / 100
            * CAV_df[(baseline_scenario, gdp_variable)]
            * money_to_currency
        )
    else:
        CAV_df["Policy_cost"] = reported_loss * money_to_currency

else:
    CAV_df["Policy_cost"] = (
        CAV_df[(baseline_scenario, gdp_variable)]
        - CAV_df[(policy_scenario, gdp_variable)]
    ) * money_to_currency

CAV_df["GHG_reduction"] = (
    CAV_df[(baseline_scenario, CAV_emissions)]
    - CAV_df[(policy_scenario, CAV_emissions)]
) * emissions_to_tonnes

CAV_df["Carbon_price"] = CAV_df[(policy_scenario, "Price|Carbon")]

CAV_df["CAV"] = CAV_df["Policy_cost"] / (
    CAV_df["GHG_reduction"] * CAV_df["Carbon_price"]
).replace(0, np.nan)

print("Cost input:", CAV_cost)
print("Emissions input:", CAV_emissions)

CAV_df.head()